### Documentation du module de NLP et Tutoriel Jupyter : classification de films avec le pipeline NLP Naive Bayes / Regression logistique combiner avec le module d'optimisation et sélection de modèles

## Pipeline Complet: NLP + Classification Multilabel

**Étapes:**
1. **Prétraitement du texte (X)**: Tokenization → Suppression stopwords → TF-IDF
2. **Encodage des labels (y)**: Convertir chaque genre en colonne binaire (multilabel binarization)
3. **Entraînement**: Utiliser LogisticRegression avec OneVsRest (approche standard pour multilabel)
4. **Évaluation**: Metrics adaptées au multilabel (Hamming Loss, Precision, Recall, F1)

Dataset link : https://www.kaggle.com/datasets/kishoreramb/movies-dataset

In [1]:
import sys 
sys.path.append(r"E:\cours ifri\Programmation et BD\Python\Pdf et tpcours\Concepts et Application\Apprentissage Automatique\ifri_mini_ml_lib")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ifri_mini_ml_lib.preprocessing.preparation.encoding import OneHotEncoder
from ifri_mini_ml_lib.preprocessing.text.test_stopwords import StopWordRemover
from ifri_mini_ml_lib.preprocessing.preparation.tf_idf import TFIDFVectorizer 
from ifri_mini_ml_lib.preprocessing.preparation.tokenization import Tokenizer
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter
from ifri_mini_ml_lib.preprocessing.text.stop_word import StopWordRemover
from ifri_mini_ml_lib.preprocessing.preparation.tf_idf import TFIDFVectorizer


# Modeles de Classification
from ifri_mini_ml_lib.classification.logistic_regression import LogisticRegression

# Cross Validation 
from ifri_mini_ml_lib.model_selection.cross_validation import k_fold_cross_validation


In [3]:
df = pd.read_csv("movies.csv" , usecols=["genres" , "overview"])



df.dropna(inplace=True)

In [4]:
df.isna().sum()

genres      0
overview    0
dtype: int64

In [5]:
# Petite modification de formatage 
df['genres'] = df['genres'].str.replace(
    "Science Fiction",
    "Science_Fiction"
)

In [6]:
splitter = DataSplitter(seed=42)


X = df[["overview"]]
y = df["genres"]




X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.2)

X_train, X_test, y_train, y_test = pd.DataFrame(X_train) , pd.DataFrame(X_test) , pd.DataFrame(y_train), pd.DataFrame(y_test)
# Proportions:
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 3818, Test: 954


In [7]:
# Pipeline NLP simple: Tokenization → Stopwords → TF-IDF

# 1. Tokenizer
tokenizer = Tokenizer()

# 2. Stop words remover

stopwords_remover = StopWordRemover(language='english')

# 3. TF-IDF

tfidf = TFIDFVectorizer()

# Utilisation:
# X_train_tokens = [tokenizer.tokenize(text) for text in X_train]  # Tokenize chaque texte
# X_train_cleaned = [stopwords_remover.transform(tokens) for tokens in X_train_tokens]  # Remove stopwords
# X_train_tfidf = tfidf.fit_transform(X_train_cleaned)  # TF-IDF


In [8]:
X.describe()

,overview
count,4772
unique,4772
top,"In the 22nd century, a paraplegic Marine is di..."
freq,1


In [9]:
X_train_tokens = X_train['overview'].apply(tokenizer.tokenize)
X_train_tokens.head()


2922    [when, rachel, phelps, inherits, the, clevelan...
3699    [the, filmed, adaptation, from, david, benioff...
113     [returning, for, his, fifth, year, of, study, ...
4720    [nat, turner, a, former, slave, in, america, l...
1374    [three, detectives, in, the, corrupt, and, bru...
Name: overview, dtype: object

In [10]:
X_train_without_stop_words = X_train_tokens.apply(stopwords_remover.fit_transform)
X_train_without_stop_words.head()

2922    [rachel, phelps, inherits, cleveland, indians,...
3699    [filmed, adaptation, david, benioff, novel, na...
113     [returning, fifth, year, study, hogwarts, harr...
4720    [nat, turner, former, slave, america, leads, l...
1374    [three, detectives, corrupt, brutal, l, police...
Name: overview, dtype: object

In [11]:
X_train_encoded = pd.DataFrame(tfidf.fit_transform(X_train_without_stop_words.tolist()))
X_train_encoded.head()

,0,1,2,3,4,5,6,7,8,9,...,18847,18848,18849,18850,18851,18852,18853,18854,18855,18856
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
y[0]

'Action Adventure Fantasy Science_Fiction'

In [13]:
# Observez la structure des genres
print("=== OBSERVATION DES GENRES ===\n")

# Exemples de genres bruts
print("Exemples de genres (premiers 10):")
print(y.head(10))

print(f"\nNombre total de films: {len(y)}")
print(f"Type de données: {type(y.iloc[0])}")

# Tous les genres uniques
all_genres_str = ' '.join(y.values)
unique_genres = set(all_genres_str.split())
print(f"\nGenres uniques totaux: {sorted(unique_genres)}")
print(f"Nombre de genres uniques: {len(unique_genres)}")

# Nombre de genres par film
y_genre_count = y.str.split().apply(len)
print(f"\nNombre de genres par film:")
print(y_genre_count.value_counts().sort_index())

=== OBSERVATION DES GENRES ===

Exemples de genres (premiers 10):
0    Action Adventure Fantasy Science_Fiction
1                    Adventure Fantasy Action
2                      Action Adventure Crime
3                 Action Crime Drama Thriller
4            Action Adventure Science_Fiction
5                    Fantasy Action Adventure
6                            Animation Family
7            Action Adventure Science_Fiction
8                    Adventure Fantasy Family
9                    Action Adventure Fantasy
Name: genres, dtype: object

Nombre total de films: 4772
Type de données: <class 'str'>

Genres uniques totaux: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'History', 'Horror', 'Movie', 'Music', 'Mystery', 'Romance', 'Science_Fiction', 'TV', 'Thriller', 'War', 'Western']
Nombre de genres uniques: 21

Nombre de genres par film:
genres
1     897
2    1489
3    1524
4     634
5     227
6       1
Name: coun

In [14]:
# Faire le prétraitement du test de la meme facon que le train


# Tokenizer -> StopWords -> TFIDFVectorizer


X_test_tokens = X_test['overview'].apply(tokenizer.tokenize)

X_test_without_stop_words = X_test_tokens.apply(stopwords_remover.transform)

X_test_encoded = pd.DataFrame(tfidf.transform(X_test_without_stop_words.tolist()))

print(f"X_test shape: {X_test_encoded.shape}")
print(f"X_test: {len(X_test_encoded)}")


X_test shape: (954, 18857)
X_test: 954


In [15]:
# ÉTAPE 2: Encodage Multilabel des genres (y_train, y_test) 
label_to_idx = {
    label: idx
    for idx, label in enumerate(unique_genres)
}
y_train_multilabel = np.zeros(
    (y_train.shape[0] , len(unique_genres)),
    dtype=np.int8
)

for idx , line in enumerate(y_train.values):
    print(f"IDX = {idx} , line = {line}")
    for genre in line[0].split():

        j = label_to_idx[genre]

        y_train_multilabel[idx, j] = 1
        
y_train_multilabel[:10]

IDX = 0 , line = ['Comedy']
IDX = 1 , line = ['Crime Drama']
IDX = 2 , line = ['Adventure Fantasy Family Mystery']
IDX = 3 , line = ['Drama']
IDX = 4 , line = ['Crime Drama Mystery Thriller']
IDX = 5 , line = ['Adventure Animation Family Fantasy']
IDX = 6 , line = ['Crime Drama Horror Thriller']
IDX = 7 , line = ['Documentary']
IDX = 8 , line = ['Drama']
IDX = 9 , line = ['Comedy Music Romance']
IDX = 10 , line = ['Fantasy Drama Comedy Family']
IDX = 11 , line = ['Drama']
IDX = 12 , line = ['Horror Thriller']
IDX = 13 , line = ['Drama History']
IDX = 14 , line = ['Action Adventure Science_Fiction Thriller']
IDX = 15 , line = ['Drama']
IDX = 16 , line = ['Action Fantasy Horror']
IDX = 17 , line = ['Action Drama Thriller']
IDX = 18 , line = ['Drama Romance']
IDX = 19 , line = ['Animation Music Family']
IDX = 20 , line = ['Science_Fiction Drama']
IDX = 21 , line = ['Drama']
IDX = 22 , line = ['Comedy Adventure Romance']
IDX = 23 , line = ['Romance Drama Comedy Music']
IDX = 24 , line = ['

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]],
      dtype=int8)

In [16]:
#Encodage de test selon la meme technique
y_test_multilabel = np.zeros(
    (y_test.shape[0] , len(unique_genres)),
    dtype=np.int8
)

for idx , line in enumerate(y_test.values):
    print(f"IDX = {idx} , line = {line}")
    for genre in line[0].split():

        j = label_to_idx[genre]

        y_test_multilabel[idx, j] = 1
        
y_test_multilabel[:10]


IDX = 0 , line = ['Action Adventure Thriller War']
IDX = 1 , line = ['Adventure Fantasy Action']
IDX = 2 , line = ['Fantasy Comedy Family']
IDX = 3 , line = ['Adventure Action Science_Fiction Thriller']
IDX = 4 , line = ['Family Animation Adventure Comedy']
IDX = 5 , line = ['Action Adventure Fantasy']
IDX = 6 , line = ['Drama']
IDX = 7 , line = ['Adventure Action Fantasy']
IDX = 8 , line = ['Comedy']
IDX = 9 , line = ['Drama Fantasy Comedy']
IDX = 10 , line = ['Comedy Drama']
IDX = 11 , line = ['Action Science_Fiction Adventure Comedy Family']
IDX = 12 , line = ['Action Crime Thriller']
IDX = 13 , line = ['Comedy Drama']
IDX = 14 , line = ['Music Drama']
IDX = 15 , line = ['Drama Comedy Romance']
IDX = 16 , line = ['Action Adventure Thriller']
IDX = 17 , line = ['Action Thriller Crime']
IDX = 18 , line = ['Horror Thriller']
IDX = 19 , line = ['Animation Adventure Family']
IDX = 20 , line = ['Action Fantasy Thriller']
IDX = 21 , line = ['Drama War']
IDX = 22 , line = ['Horror Drama Sci

array([[1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0]],
      dtype=int8)

In [17]:
#Codons la hamming_Loss

def hamming_loss(y_true : np.ndarray , y_pred : np.ndarray):
    # 1. Trouver où les valeurs sont différentes (True si erreur, False si correct)
    erreurs = y_true != y_pred

    # 2. Calculer la moyenne de ces erreurs
    hamming_loss_numpy = np.mean(erreurs)

    return hamming_loss_numpy


In [18]:
X_train_encoded.shape

(3818, 18857)

In [ ]:
# ÉTAPE 3: Classification Multilabel
X_train_encoded = X_train_encoded
y_train_encoded = y_train_multilabel
print("ÉTAPE 3: Entraînement des classifieurs Multilabel (Un model pour chaque classe)")

idx_to_label = {
    idx: label
    for idx, label in enumerate(unique_genres)
}
#Nombre de classes a predire
TARGETS = len(unique_genres)

models  = []

for j in range(TARGETS):
    print(f"Entrainement Numero {j} --> Genre : {idx_to_label[j]}")

    y_label =y_train_encoded[:, j]   # colonne du genre j

    model = LogisticRegression()  # logistic regression 

    score = k_fold_cross_validation(model=model , X= X_train_encoded , y=y_label , metric=hamming_loss , stratified=True)
    
    models.append(model)
    print(f"Hamming Loss pour {idx_to_label[j]} = {hamming_loss(y_label , model.predict(X_train_encoded))} ")

ÉTAPE 3: Entraînement des classifieurs Multilabel (Un model pour chaque classe)
Entrainement Numero 0 --> Genre : Action
Hamming Loss pour Action = 0.2388685175484547 
Entrainement Numero 1 --> Genre : TV
